In [1]:
import sys
#!{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir transformers sentence-transformers

# On met à jour Pandas pour qu'il s'aligne avec le reste
#!{sys.executable} -m pip install --upgrade pandas "numpy<2.0"

#!{sys.executable} -m pip install gensim

In [2]:
#!pip install --upgrade transformers sentence-transformers
#!pip install pypdf
import os
import re
import json
import math
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PDF ingestion
from pypdf import PdfReader

# Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec

# (Optional) local generator
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Distributed computing system. 
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession 
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    LongType, FloatType, ArrayType
)

/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [3]:
@dataclass
class RAGConfig:
    # --- Chemins et exécution (Paramètres exigés par le PDF) ---
    input_dir: str = "course_material"
    output_dir: str = "project/embeddings"
    parallelism: int = 4

    # --- Chunking (simple word-based chunking) ---
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    # --- Retrieval ---
    top_k: int = 4
    similarity_metric: str = "cosine"   # {"cosine", "euclidean"}
    embedding_strategy: str = "word2vec"   # {"tfidf", "sbert", "word2vec"}

    # --- Embedding models ---
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # --- Generator (fallback: small, CPU-friendly) ---
    gen_model_name: str = "google/flan-t5-small"
    max_new_tokens: int = 256

cfg = RAGConfig()

In [4]:
# Init Park
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()
    print("SparkContext existant arrêté.")
    
conf = SparkConf() \
    .setAppName("IngestData") \
    .setMaster("local[*]")

sc = SparkContext(conf=conf)
print("Spark version:", sc.version)

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 13:36:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.2.1


In [5]:
def get_paths_pdf_files(folder_path: str) -> List[str]:
    """
    Browse a folder and get path from each PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.

    Returns
    -------
    results : list of string
    """

    folder = Path(folder_path)

    # return paths all pdf files in folder path
    return list( folder.glob("*.pdf") )

# --- quick sanity check (should not error once implemented) ---
pdf_paths = get_paths_pdf_files(cfg.input_dir)
rdd = sc.parallelize(pdf_paths, numSlices=cfg.parallelism)
print(pdf_paths)

[PosixPath('course_material/1-intro.pdf'), PosixPath('course_material/2-spark.pdf'), PosixPath('course_material/3-stream_proc.pdf'), PosixPath('course_material/4-stream_algos.pdf'), PosixPath('course_material/5-nosql.pdf'), PosixPath('course_material/6-parallel.pdf'), PosixPath('course_material/9780521192248_excerpt.pdf'), PosixPath('course_material/A Neural Network Playground.pdf'), PosixPath('course_material/a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf'), PosixPath('course_material/a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf'), PosixPath('course_material/a_the-big-book-of-machine-learning-use-cases.pdf'), PosixPath('course_material/a_TowardScalableSystemsForBigDataAnalytics.pdf'), PosixPath('course_material/Finding Similar Items.pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted (1).pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted.pdf'), PosixPath('course_material/intro.pdf'), PosixP

In [6]:
def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    
    import logging
    logging.getLogger("pypdf").setLevel(logging.ERROR)
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : Path(pdf_path).name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages

# flatMap : Regroupe tout les pages de chaque pdf dans une seule liste
pages_rdd = rdd.flatMap(extract_pages_from_pdf)

In [7]:
def clean_text(page_from_pdf: dict) -> dict:
    """Basic cleanup for PDF-extracted text."""
    text = page_from_pdf["text"]
    # 1. Supprimer les césures en fin de ligne
    text = text.replace("-\n", "")
    
    # 2. Remplacer les retours à la ligne restants par des espaces
    text = text.replace("\n", " ")
    
    # 3. Réduire les espaces multiples à un seul espace
    text = re.sub(r"\s+", " ", text)
    
    # 4. Supprimer les espaces en début et fin
    page_from_pdf["text"] = text.strip()
    
    return page_from_pdf

# map : traitement chaque page
clean_rdd = pages_rdd.map(clean_text)
print(clean_rdd.count(), "pages extraites")
print(clean_rdd.first())

1662 pages extraites
{'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'}


In [8]:
def word_based_chunk_page(page_from_pdf: dict) -> list:
    """Split one page into overlapping word chunks."""
    
    chunk_size_words = cfg.chunk_size_words 
    overlap_words = cfg.chunk_overlap_words

    words = page_from_pdf["text"].split()
    page_num = page_from_pdf["page_num"]
    
    chunks = []
    step = chunk_size_words - overlap_words
    
    if step <= 0:
        raise ValueError("overlap_words must be smaller than chunk_size_words")
    
    chunk_index = 0
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id": f"p{page_num}_c{chunk_index}",
            "pdf_name": page_from_pdf["pdf_name"],
            "page_num":  page_num,
            "text": " ".join(chunk_words),
            "start_word": start,        
            "end_word": end - 1
        })
        chunk_index += 1
        if end >= len(words):
            break
    return chunks
# Build chunks
clean_chunk_rdd = clean_rdd.flatMap(word_based_chunk_page)
print(clean_chunk_rdd.count(), " chunks")
print(clean_chunk_rdd.first())

2982  chunks
{'chunk_id': 'p1_c0', 'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis', 'start_word': 0, 'end_word': 199}


In [9]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [10]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, Word2Vec as SparkWord2Vec
from pyspark.sql.functions import udf, lit

# 1. Initialisation propre de la SparkSession
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

# 2. Création du DataFrame de base avec les IDs et métadonnées
rdd_base = clean_chunk_rdd.zipWithIndex().map(
    lambda x: (
        x[1],                        # id (LongType)
        x[0]["pdf_name"],            # source_pdf
        x[0]["page_num"],            # page_num
        x[0]["chunk_id"],            # chunk_id
        x[0]["start_word"],          # start_word (Entier)
        x[0]["end_word"],            # end_word (Entier)
        x[0]["text"],                # chunk_text
    )
)

schema_base = StructType([
    StructField("id", LongType(), False),
    StructField("source_pdf", StringType(), True),
    StructField("page_num", IntegerType(), True),
    StructField("chunk_id", StringType(), True),
    StructField("start_word", IntegerType(), True),
    StructField("end_word", IntegerType(), True),
    StructField("chunk_text", StringType(), True),
])

df_base = spark.createDataFrame(rdd_base, schema=schema_base)

# Le schéma FINAL exigé (avec les colonnes d'embeddings)
final_schema = StructType(schema_base.fields + [
    StructField("embedding", ArrayType(FloatType()), True),
    StructField("embedding_strategy", StringType(), True)
])

df_final = None # Ce sera notre DataFrame prêt à être sauvegardé

# =====================================================================
# 3. CHOIX DE LA STRATÉGIE D'EMBEDDING
# =====================================================================

if cfg.embedding_strategy == "sbert":
    # --- STRATÉGIE 1 : SBERT (via mapPartitions hermétique) ---
    def embed_partition_sbert(partition):
        from sentence_transformers import SentenceTransformer
        import numpy as np
        
        model = SentenceTransformer(cfg.sbert_model_name)
        for row in partition:
            row_dict = row.asDict()
            emb = model.encode(row_dict["chunk_text"], normalize_embeddings=True)
            row_dict["embedding"] = np.asarray(emb, dtype=np.float32).tolist()
            row_dict["embedding_strategy"] = "sbert"
            yield row_dict

    rdd_embedded = df_base.rdd.mapPartitions(embed_partition_sbert)
    df_final = spark.createDataFrame(rdd_embedded, schema=final_schema)

elif cfg.embedding_strategy == "tfidf":
    # --- STRATÉGIE 2 : TF-IDF (Natif Spark MLlib) ---
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
    wordsData = tokenizer.transform(df_base)
    
    hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20000)
    featurizedData = hashingTF.transform(wordsData)
    
    idf = IDF(inputCol="rawFeatures", outputCol="features")
    idfModel = idf.fit(featurizedData)
    rescaledData = idfModel.transform(featurizedData)
    
    vector_to_array = udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
    
    df_final = rescaledData.withColumn("embedding", vector_to_array("features")) \
                           .withColumn("embedding_strategy", lit("tfidf")) \
                           .drop("words", "rawFeatures", "features")
    idfModel.write().overwrite().save("/home/guest/shared/course_material/tfidf_model.model")

elif cfg.embedding_strategy == "word2vec":
    # --- STRATÉGIE 3 : Word2Vec (Natif Spark MLlib) ---
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
    wordsData = tokenizer.transform(df_base)
    
    word2Vec = SparkWord2Vec(vectorSize=100, minCount=2, inputCol="words", outputCol="features")
    model = word2Vec.fit(wordsData)
    w2vData = model.transform(wordsData)
    
    vector_to_array = udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
    
    df_final = w2vData.withColumn("embedding", vector_to_array("features")) \
                      .withColumn("embedding_strategy", lit("word2vec")) \
                      .drop("words", "features")
# 1. Sauvegarde du Cerveau (Le modèle pour le Notebook 2)
    model.write().overwrite().save("/home/guest/shared/course_material/w2v_model.model")
    
# =====================================================================
# 4. SAUVEGARDE ET TEST
# =====================================================================
if df_final is not None:
    import os
    
    # 1. On sauvegarde d'abord dans un dossier 100% Linux (pas de conflit avec ton PC)
    linux_internal_path = f"/home/guest/embeddings_{cfg.embedding_strategy}.parquet"
    df_final.write.mode("overwrite").parquet(linux_internal_path)
    print(f" Sauvegarde réussie dans le conteneur : {linux_internal_path}")
    
    # 2. On copie manuellement le résultat fini vers ton dossier partagé
    shared_path = f"/home/guest/shared/course_material/embeddings_{cfg.embedding_strategy}.parquet"
    os.system(f"cp -r {linux_internal_path} {shared_path}")
    print(f" Fichiers copiés vers ton PC : {shared_path}\n")
    
    # 3. Test de lecture (depuis le dossier partagé)
    print("="*70)
    print("VÉRIFICATION : LECTURE DU FICHIER PARQUET GÉNÉRÉ")
    print("="*70)
    test_df = spark.read.parquet(shared_path)
    print(f"Nombre de chunks sauvegardés et relus : {test_df.count()}")
    test_df.select("id", "source_pdf", "chunk_id", "start_word", "end_word", "embedding_strategy").show(3)

26/05/21 13:38:20 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/21 13:38:20 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


 Sauvegarde réussie dans le conteneur : /home/guest/embeddings_word2vec.parquet
 Fichiers copiés vers ton PC : /home/guest/shared/course_material/embeddings_word2vec.parquet

VÉRIFICATION : LECTURE DU FICHIER PARQUET GÉNÉRÉ
Nombre de chunks sauvegardés et relus : 2982
+---+--------------------+--------+----------+--------+------------------+
| id|          source_pdf|chunk_id|start_word|end_word|embedding_strategy|
+---+--------------------+--------+----------+--------+------------------+
|627|A Neural Network ...|   p1_c0|         0|     199|          word2vec|
|628|A Neural Network ...|   p1_c1|       150|     349|          word2vec|
|629|A Neural Network ...|   p2_c0|         0|     199|          word2vec|
+---+--------------------+--------+----------+--------+------------------+
only showing top 3 rows



In [11]:
import os
chemin = f"/home/guest/shared/course_material/embeddings_{cfg.embedding_strategy}.parquet"
print("Fichiers trouvés dans le dossier :")
print(os.listdir(chemin))

Fichiers trouvés dans le dossier :
['.part-00000-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet.crc', '.part-00001-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet.crc', '.part-00002-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet.crc', '.part-00003-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet', 'part-00001-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet', 'part-00002-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet', 'part-00003-8c04a4b5-040a-4906-8be3-1a1067335367-c000.snappy.parquet', '_SUCCESS']


In [12]:
print("ok")

ok


In [13]:
# On relit le parquet avec Spark
df_test = spark.read.parquet("/home/guest/embeddings_sbert.parquet")

# On prend juste 5 lignes, on sélectionne quelques colonnes utiles, et on le convertit en Pandas pour un bel affichage
affichage = df_test.select("source_pdf", "page_num", "chunk_text").limit(5).toPandas()

# On affiche le tableau
display(affichage)

,source_pdf,page_num,chunk_text
0,A Neural Network Playground.pdf,1,"Epoch 000,000 Learning rateActivation Regulari..."
1,A Neural Network Playground.pdf,1,of trainingto testdata: 50% Noise: 0 Batch siz...
2,A Neural Network Playground.pdf,2,"This Is Cool, Can I Repurpose It? Please do! W..."
3,A Neural Network Playground.pdf,2,weights of theconnections between neurons. Blu...
4,A Neural Network Playground.pdf,3,Source on GitHub (https://github.com/tensorflo...
